# Notebook 13: Circuit Comparison & Motif Detection
**Discovering Universal Computational Patterns Across Neural Networks**

## What You'll Learn

In this notebook, you'll explore how to compare circuits and find recurring patterns:

- **Cross-Model Circuit Comparison**: Compare circuits from different architectures
- **Similarity Metrics**: Quantifying structural and functional similarity
- **Motif Detection**: Finding recurring computational subgraph patterns
- **Statistical Significance**: Distinguishing real patterns from noise
- **Consensus Circuits**: Extracting universal computational structures

**Prerequisites**: 
- Notebook 11 (Path Patching & ACDC)
- Basic graph theory
- Understanding of statistical testing

**Time Estimate**: 60-90 minutes

---

## Background: Why Compare Circuits?

### The Universality Question

**Key Questions**:
1. Do different models learn the same algorithm?
2. Are there universal computational patterns?
3. Which circuit components are task-specific vs generic?
4. How does architecture affect learned circuits?

### Circuit Comparison Applications

**Transfer Learning**: Identify transferable circuit components

**Model Debugging**: Compare correct vs buggy model circuits

**Architecture Search**: Find architectures that encourage beneficial circuits

**Interpretability**: Discover fundamental computational primitives

### Computational Motifs

Like biological motifs (e.g., transcription factor networks), neural circuits contain **recurring subgraph patterns**:

**Common Motifs**:
- **Feedforward**: A → B → C (information cascade)
- **Recurrent**: A → B → A (feedback loop)
- **Skip Connection**: A → C (bypassing B)
- **Convergent**: A → C, B → C (information aggregation)
- **Divergent**: A → B, A → C (information broadcasting)
- **Triangle**: A → B, B → C, A → C (redundant paths)

### Key Papers

- **Milo et al. (2002)**: "Network Motifs: Simple Building Blocks of Complex Networks"
- **Fodor & Pylyshyn (1988)**: "Connectionism and cognitive architecture"
- **Olah et al. (2020)**: "Zoom In: An Introduction to Circuits"

---

In [ ]:
# Imports
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import neuros-mechint circuit tools
from neuros_mechint.circuits import (
    AutomatedCircuitDiscovery,
    CircuitComparator,
    MotifDetector,
    Edge,
    Circuit
)
from neuros_mechint.database import MechIntDatabase

print("✓ All imports successful!")

---

## Part 1: Training Multiple Models on Same Task

To compare circuits, we first need multiple models. We'll train 3 different architectures on the same task (pattern detection from Notebook 11).

**Architectures**:
1. **Wide-Shallow**: Few layers, many neurons
2. **Deep-Narrow**: Many layers, few neurons  
3. **Balanced**: Medium layers, medium neurons

**Hypothesis**: Different architectures may discover different circuits for the same task.

---

In [ ]:
# Define three different architectures
class WideShallow(nn.Module):
    """Wide but shallow network."""
    def __init__(self, input_size=784, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

class DeepNarrow(nn.Module):
    """Deep but narrow network."""
    def __init__(self, input_size=784, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

class Balanced(nn.Module):
    """Balanced depth and width."""
    def __init__(self, input_size=784, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

# Create all three models
models = {
    'Wide-Shallow': WideShallow().to(device),
    'Deep-Narrow': DeepNarrow().to(device),
    'Balanced': Balanced().to(device)
}

print("Created 3 architectures:")
for name, model in models.items():
    n_params = sum(p.numel() for p in model.parameters())
    n_layers = len([m for m in model.modules() if isinstance(m, nn.Linear)])
    print(f"  {name:15s}: {n_layers} layers, {n_params:,} parameters")

In [ ]:
# Generate training data (binary classification)
def generate_binary_data(n_samples=1000, input_dim=784):
    """Generate simple binary classification data."""
    # Create two clusters
    n_per_class = n_samples // 2
    
    # Class 0: centered at -1
    class0 = torch.randn(n_per_class, input_dim) * 0.5 - 1.0
    # Class 1: centered at +1  
    class1 = torch.randn(n_per_class, input_dim) * 0.5 + 1.0
    
    data = torch.cat([class0, class1], dim=0)
    labels = torch.cat([torch.zeros(n_per_class), torch.ones(n_per_class)]).long()
    
    # Shuffle
    perm = torch.randperm(len(data))
    return data[perm], labels[perm]

# Generate datasets
train_data, train_labels = generate_binary_data(n_samples=800)
test_data, test_labels = generate_binary_data(n_samples=200)

print(f"Training data: {train_data.shape}")
print(f"Test data: {test_data.shape}")
print(f"Class balance: {train_labels.float().mean():.2%} positive")

In [ ]:
# Train all models (quick training for demo)
def train_model(model, train_data, train_labels, epochs=30, lr=1e-3):
    """Quick training loop."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(train_data.to(device))
        loss = criterion(outputs, train_labels.to(device))
        loss.backward()
        optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        test_outputs = model(test_data.to(device))
        test_preds = test_outputs.argmax(dim=1).cpu()
        accuracy = (test_preds == test_labels).float().mean().item()
    
    return accuracy

# Train all models
print("Training models...\n")
accuracies = {}

for name, model in models.items():
    acc = train_model(model, train_data, train_labels)
    accuracies[name] = acc
    print(f"{name:15s}: {acc:.2%} test accuracy")

print("\n✓ All models trained successfully!")

---

## Part 2: Extracting Circuits from Each Model

Now we'll use ACDC to extract the minimal computational circuit from each model.

**Goal**: Discover what circuit each architecture learned for the same task.

---

In [ ]:
# Extract circuits using ACDC
print("Extracting circuits...\n")

circuits = {}
sample_inputs = test_data[:32].to(device)
sample_labels = test_labels[:32].to(device)

for name, model in models.items():
    print(f"Running ACDC on {name}...")
    
    acdc = AutomatedCircuitDiscovery(
        model=model,
        importance_threshold=0.01,
        device=device
    )
    
    circuit = acdc.discover_circuit(
        inputs=sample_inputs,
        targets=sample_labels,
        max_iterations=15
    )
    
    circuits[name] = circuit
    print(f"  → {len(circuit.edges)} edges, {len(circuit.nodes)} nodes\n")

print("✓ Circuits extracted!")

In [ ]:
# Visualize circuit sizes
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names = list(circuits.keys())
n_edges = [len(c.edges) for c in circuits.values()]
n_nodes = [len(c.nodes) for c in circuits.values()]
avg_importance = [c.average_importance for c in circuits.values()]

# Bar plot: Circuit size
x = np.arange(len(names))
width = 0.35

axes[0].bar(x - width/2, n_edges, width, label='Edges', alpha=0.7)
axes[0].bar(x + width/2, n_nodes, width, label='Nodes', alpha=0.7)
axes[0].set_xlabel('Model Architecture')
axes[0].set_ylabel('Count')
axes[0].set_title('Circuit Size Comparison', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=15)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Bar plot: Average importance
axes[1].bar(names, avg_importance, alpha=0.7, color='purple')
axes[1].set_xlabel('Model Architecture')
axes[1].set_ylabel('Average Edge Importance')
axes[1].set_title('Circuit Edge Importance', fontweight='bold')
axes[1].set_xticklabels(names, rotation=15)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nObservation:")
print("  Different architectures → different circuit sizes")
print("  But do they compute the same algorithm?")

---

## Part 3: Circuit Comparison

### Similarity Metrics

How do we compare two circuits? Multiple metrics:

**1. Node Overlap**:
$$S_{\text{node}} = \frac{|N_A \cap N_B|}{|N_A \cup N_B|}$$

**2. Edge Overlap**:
$$S_{\text{edge}} = \frac{|E_A \cap E_B|}{|E_A \cup E_B|}$$

**3. Structural Similarity** (graph isomorphism):
Check if circuits have the same topology (ignoring labels).

**4. Functional Similarity** (correlation of outputs):
$$S_{\text{func}} = \text{corr}(f_A(X), f_B(X))$$

---

In [ ]:
# Initialize database to store circuits
import tempfile
temp_dir = tempfile.mkdtemp()

db = MechIntDatabase(root_dir=temp_dir)

# Store all circuits in database
circuit_ids = {}
for name, circuit in circuits.items():
    # Convert to MechIntResult
    from neuros_mechint.results import CircuitResult
    result = CircuitResult(
        method='ACDC',
        data=circuit,
        metadata={'model_name': name, 'task': 'binary_classification'},
        metrics={'n_edges': len(circuit.edges), 'n_nodes': len(circuit.nodes)}
    )
    
    circuit_id = db.store(result, tags=[name, 'acdc', 'binary_task'])
    circuit_ids[name] = circuit_id
    print(f"Stored {name} circuit: {circuit_id[:8]}...")

print(f"\n✓ All circuits stored in database")

In [ ]:
# Initialize Circuit Comparator
comparator = CircuitComparator(database=db)

print("CircuitComparator initialized:")
print(f"  - Database: {len(circuit_ids)} circuits")
print(f"  - Ready to compare")

In [ ]:
# Pairwise circuit comparison
print("\nPairwise Circuit Comparisons:\n")
print("="*70)

comparisons = {}
model_names = list(circuit_ids.keys())

for i, name_a in enumerate(model_names):
    for name_b in model_names[i+1:]:
        print(f"\n{name_a} vs {name_b}:")
        
        comparison = comparator.compare_circuits(
            circuit_a_id=circuit_ids[name_a],
            circuit_b_id=circuit_ids[name_b]
        )
        
        comparisons[(name_a, name_b)] = comparison
        
        print(f"  Overall Similarity:     {comparison.similarity_score:.3f}")
        print(f"  Node Overlap:          {comparison.node_overlap:.3f}")
        print(f"  Edge Overlap:          {comparison.edge_overlap:.3f}")
        print(f"  Structural Similarity: {comparison.structural_similarity:.3f}")
        print(f"  Common Nodes:          {len(comparison.common_nodes)}")
        print(f"  Common Edges:          {len(comparison.common_edges)}")

print("\n" + "="*70)
print("\n✓ All pairwise comparisons complete!")

In [ ]:
# Visualize similarity matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Create similarity matrices
n_models = len(model_names)
sim_matrix = np.eye(n_models)  # Diagonal is 1 (self-similarity)
struct_matrix = np.eye(n_models)

for (name_a, name_b), comp in comparisons.items():
    i = model_names.index(name_a)
    j = model_names.index(name_b)
    
    sim_matrix[i, j] = comp.similarity_score
    sim_matrix[j, i] = comp.similarity_score
    
    struct_matrix[i, j] = comp.structural_similarity
    struct_matrix[j, i] = comp.structural_similarity

# Plot 1: Overall similarity
im1 = axes[0].imshow(sim_matrix, cmap='YlGnBu', vmin=0, vmax=1)
axes[0].set_xticks(range(n_models))
axes[0].set_yticks(range(n_models))
axes[0].set_xticklabels(model_names, rotation=45, ha='right')
axes[0].set_yticklabels(model_names)
axes[0].set_title('Overall Circuit Similarity', fontweight='bold', fontsize=12)

# Add values
for i in range(n_models):
    for j in range(n_models):
        text = axes[0].text(j, i, f'{sim_matrix[i, j]:.2f}',
                           ha="center", va="center", color="black", fontsize=10)

plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

# Plot 2: Structural similarity
im2 = axes[1].imshow(struct_matrix, cmap='RdPu', vmin=0, vmax=1)
axes[1].set_xticks(range(n_models))
axes[1].set_yticks(range(n_models))
axes[1].set_xticklabels(model_names, rotation=45, ha='right')
axes[1].set_yticklabels(model_names)
axes[1].set_title('Structural Similarity', fontweight='bold', fontsize=12)

# Add values
for i in range(n_models):
    for j in range(n_models):
        text = axes[1].text(j, i, f'{struct_matrix[i, j]:.2f}',
                           ha="center", va="center", color="black", fontsize=10)

plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Warmer colors = higher similarity")
print("  - Diagonal is always 1.0 (self-similarity)")
print("  - Different architectures can have similar circuits!")

---

## Part 4: Motif Detection

### Finding Recurring Patterns

**Computational Motifs** are small subgraph patterns that appear more frequently than expected by chance.

**6 Common Motifs**:
1. **Feedforward**: A → B → C (cascade)
2. **Recurrent**: A → B, B → A (feedback)
3. **Skip**: A → B, A → C, B → C (shortcut)
4. **Convergent**: A → C, B → C (merge)
5. **Divergent**: A → B, A → C (broadcast)
6. **Triangle**: A → B, B → C, C → A (cycle)

### Statistical Significance

To determine if a motif is significant:
1. Count occurrences in real circuit
2. Generate random circuits (same size)
3. Count occurrences in random circuits
4. Compute Z-score:

$$Z = \frac{N_{\text{real}} - \langle N_{\text{random}} \rangle}{\sigma_{\text{random}}}$$

$Z > 2$ typically indicates significance.

---

In [ ]:
# Detect motifs in each circuit
print("Detecting computational motifs...\n")

motif_results = {}

for name, circuit in circuits.items():
    print(f"Analyzing {name}...")
    
    detector = MotifDetector(
        circuit=circuit,
        n_random_samples=50  # For significance testing
    )
    
    analysis = detector.detect_all_motifs(compute_significance=True)
    motif_results[name] = analysis
    
    print(f"  Found {analysis.total_motifs} total motif instances")
    print(f"  Significant motifs: {sum(1 for m in analysis.motif_counts.values() if analysis.z_scores.get(m.motif_type, 0) > 2)}\n")

print("✓ Motif detection complete!")

In [ ]:
# Visualize motif distributions
motif_types = ['feedforward', 'recurrent', 'skip', 'convergent', 'divergent', 'triangle']

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Extract motif counts for each model
model_motif_counts = {name: defaultdict(int) for name in model_names}
model_z_scores = {name: {} for name in model_names}

for name, analysis in motif_results.items():
    for motif_type, instances in analysis.motif_counts.items():
        model_motif_counts[name][motif_type] = len(instances)
        model_z_scores[name][motif_type] = analysis.z_scores.get(motif_type, 0.0)

# Plot 1: Motif counts
x = np.arange(len(motif_types))
width = 0.25

for i, name in enumerate(model_names):
    counts = [model_motif_counts[name][mt] for mt in motif_types]
    axes[0].bar(x + i*width, counts, width, label=name, alpha=0.7)

axes[0].set_xlabel('Motif Type')
axes[0].set_ylabel('Count')
axes[0].set_title('Motif Occurrence Across Models', fontweight='bold', fontsize=12)
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(motif_types, rotation=15)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Z-scores (significance)
for i, name in enumerate(model_names):
    z_scores = [model_z_scores[name].get(mt, 0.0) for mt in motif_types]
    axes[1].bar(x + i*width, z_scores, width, label=name, alpha=0.7)

axes[1].axhline(y=2, linestyle='--', color='red', linewidth=2, alpha=0.7, label='Significance threshold')
axes[1].set_xlabel('Motif Type')
axes[1].set_ylabel('Z-Score')
axes[1].set_title('Motif Statistical Significance', fontweight='bold', fontsize=12)
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(motif_types, rotation=15)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Bars above red line (Z > 2): Statistically significant motifs")
print("  - These patterns appear more than random chance")
print("  - May represent fundamental computational primitives")

In [ ]:
# Find consensus motifs (present in all models)
print("\nConsensus Motifs (present in all models):\n")
print("="*60)

# Find motifs with Z > 2 in all models
consensus_motifs = []

for motif_type in motif_types:
    significant_in_all = all(
        model_z_scores[name].get(motif_type, 0) > 2 
        for name in model_names
    )
    
    if significant_in_all:
        consensus_motifs.append(motif_type)
        avg_z = np.mean([model_z_scores[name][motif_type] for name in model_names])
        print(f"  ✓ {motif_type:15s} (avg Z-score: {avg_z:.2f})")

if not consensus_motifs:
    print("  No consensus motifs found (Z > 2 in all models)")
    print("  Try: Lower threshold or more training data")

print("="*60)
print(f"\nTotal consensus motifs: {len(consensus_motifs)}/{len(motif_types)}")

---

## Summary & Key Takeaways

### What You Learned

1. **Circuit Comparison**:
   - Multiple metrics: node/edge overlap, structural, functional
   - Different architectures can learn similar circuits
   - Similarity matrices reveal cross-model patterns

2. **Motif Detection**:
   - 6 common computational motifs
   - Statistical significance via randomization tests
   - Z-scores quantify motif enrichment
   - Consensus motifs = universal patterns

3. **Universal Computation**:
   - Some circuit components are task-specific
   - Others are universal (appear across models)
   - Motifs reveal computational primitives

### Key Equations

**Node Overlap**:
$$S_{\text{node}} = \frac{|N_A \cap N_B|}{|N_A \cup N_B|}$$

**Edge Overlap**:
$$S_{\text{edge}} = \frac{|E_A \cap E_B|}{|E_A \cup E_B|}$$

**Motif Z-Score**:
$$Z = \frac{N_{\text{real}} - \langle N_{\text{random}} \rangle}{\sigma_{\text{random}}}$$

### Practical Applications

1. **Transfer Learning**: Identify transferable circuit components
2. **Model Selection**: Choose architectures that learn desired circuits
3. **Debugging**: Compare correct vs buggy model circuits
4. **Interpretability**: Map functionality to motif types
5. **Architecture Design**: Encourage beneficial motif patterns

### Next Steps

- **Notebook 15**: Energy flow through circuit motifs
- **Notebook 16**: Automate circuit analysis pipeline

### References

1. **Milo et al. (2002)**: "Network Motifs: Simple Building Blocks of Complex Networks"
2. **Olah et al. (2020)**: "Zoom In: An Introduction to Circuits"
3. **Fodor & Pylyshyn (1988)**: "Connectionism and cognitive architecture"

---

## Exercises

### Exercise 1: Different Tasks
Train models on different tasks. Do consensus motifs change?

**Starter code**:

In [ ]:
# Exercise 1: Task-dependent motifs
# TODO: Create different task (e.g., XOR vs AND)
# TODO: Extract circuits
# TODO: Compare motifs across tasks

# Your code here:
pass

### Exercise 2: Custom Motifs
Define your own motif patterns and search for them.

**Starter code**:

In [ ]:
# Exercise 2: Custom motif detection
# TODO: Define custom graph pattern
# TODO: Search circuit for pattern
# TODO: Compute significance

# Your code here:
pass

### Exercise 3: Functional Similarity
Measure how similar circuit outputs are (functional similarity).

**Starter code**:

In [ ]:
# Exercise 3: Functional similarity
# TODO: Extract circuit outputs for test data
# TODO: Compute correlation between outputs
# TODO: Compare with structural similarity

# Your code here:
pass

---

**Congratulations!** You've mastered circuit comparison and motif detection. You can now find universal computational patterns across different neural network architectures.

Continue to **Notebook 15: Energy Cascades & Hamiltonian Decomposition** to understand the energy structure of these circuits.